# SMS Spam Classification Pipeline

This notebook trains and evaluates a machine learning text-classification model to separate unwanted SMS messages () from legitimate messages ().

## Dataset Source & Attribution
- **Dataset**: [SMS Spam Collection](https://huggingface.co/datasets/codesignal/sms-spam-collection) via Hugging Face (CC BY 4.0)
- **Target**: Predict  ( vs ) from .

In [ ]:
import urllib.request
from pathlib import Path
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.pipeline import Pipeline

## 1. Data Loading & Exploratory Data Analysis

In [ ]:
data_path = Path("sms-spam-collection.csv")
if not data_path.exists():
    url = "https://huggingface.co/datasets/codesignal/sms-spam-collection/raw/main/sms-spam-collection.csv"
    urllib.request.urlretrieve(url, data_path)

df = pd.read_csv(data_path)
print(f"Total records: {len(df)}")
print("Class distribution:")
print(df["label"].value_counts(normalize=True).map("{:.2%}".format))
df.head()

## 2. Stratified Train / Test Split
Because the dataset is imbalanced (86.6% ham, 13.4% spam), we use stratified splitting to guarantee that both the training and test sets maintain the exact same proportion of spam.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label"], test_size=0.20, random_state=42, stratify=df["label"]
)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

## 3. Majority-Class Baseline
A naive model that always predicts the majority class (). Shows why accuracy alone is misleading.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")
print(classification_report(y_test, y_pred_base, zero_division=0))

## 4. Model Training: Logistic Regression & Multinomial Naive Bayes
We fit text transformations () strictly on the training set to prevent data leakage.

In [ ]:
lr_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2), sublinear_tf=True)),
    ("clf", LogisticRegression(random_state=42, class_weight="balanced"))
])
lr_pipe.fit(X_train, y_train)

nb_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2), sublinear_tf=True)),
    ("clf", MultinomialNB())
])
nb_pipe.fit(X_train, y_train)

## 5. Comparative Evaluation

In [ ]:
for name, pipe in [("Logistic Regression", lr_pipe), ("Multinomial Naive Bayes", nb_pipe)]:
    preds = pipe.predict(X_test)
    print(f"=== {name} ===")
    print(f"Accuracy:  {accuracy_score(y_test, preds):.4f}")
    print(f"Precision: {precision_score(y_test, preds, pos_label='spam'):.4f}")
    print(f"Recall:    {recall_score(y_test, preds, pos_label='spam'):.4f}")
    print(f"F1-Score:  {f1_score(y_test, preds, pos_label='spam'):.4f}")
    print(f"Confusion Matrix:{confusion_matrix(y_test, preds, labels=['ham', 'spam'])}")
    print(classification_report(y_test, preds, zero_division=0))


## 6. Error Analysis: False Positives & False Negatives

In [ ]:
results = pd.DataFrame({"message": X_test, "actual": y_test, "pred": lr_pipe.predict(X_test)})
fp = results[(results["actual"] == "ham") & (results["pred"] == "spam")]
fn = results[(results["actual"] == "spam") & (results["pred"] == "ham")]

print(f"False Positives: {len(fp)}")
for msg in fp["message"].head(3):
    print("  -", msg)

print(f"False Negatives: {len(fn)}")
for msg in fn["message"].head(3):
    print("  -", msg)

## 7. Model Persistence
Export the trained pipeline with .

In [ ]:
joblib.dump(lr_pipe, "spam_classifier.joblib")
print("Exported pipeline to spam_classifier.joblib")